In [ ]:
import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.spatial.distance import jensenshannon

# === CONFIGURATION ===
proportions_file = 'TOO-Decon-Noise/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon-Noise'
results_dir = os.path.join(root_dir, 'Results-JSD-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)

prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)

prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})

prop_df = prop_df.drop(
    columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'],
    errors='ignore'
)

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus',
    'FemaleReproductive', 'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes',
    'MuscleSkeletal', 'NerveTibial', 'Pancreas', 'Pituitary', 'Prostate', 'SalivaryGland',
    'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# Add missing columns
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

# Normalize true proportions to percentages
prop_sums = prop_df[reference_tissues].sum(axis=1)

if (prop_sums == 0).any():
    print(f"Warning: {(prop_sums == 0).sum()} true-proportion rows sum to zero.")

prop_df_tissues = (
    prop_df[reference_tissues]
    .div(prop_sums.replace(0, np.nan), axis=0)
    .multiply(100)
    .round(5)
    .fillna(0)
)

prop_df.update(prop_df_tissues)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*_500', '*_modified.txt')))

results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))
        dir_parts = dir_name.replace("Decon-Results_", "").split("_")

        noise_level = 'NA'
        for part in dir_parts:
            if part.startswith("Noise"):
                noise_level = part.replace("Noise", "")
                break

        matrix = '_'.join(dir_parts[-3:]) if len(dir_parts) >= 3 else dir_name

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0

        decon_df = decon_df[reference_tissues]

        common_samples = prop_df.index.intersection(decon_df.index)

        true_vals_df = prop_df.loc[common_samples, reference_tissues].fillna(0)
        pred_vals_df = decon_df.loc[common_samples, reference_tissues].fillna(0)

        sample_jsd = []
        skipped_samples = 0

        for sample in common_samples:
            true_vec = true_vals_df.loc[sample].values.astype(float)
            pred_vec = pred_vals_df.loc[sample].values.astype(float)

            # Ensure non-negative values
            true_vec = np.clip(true_vec, 0, None)
            pred_vec = np.clip(pred_vec, 0, None)

            true_sum = true_vec.sum()
            pred_sum = pred_vec.sum()

            # Skip samples where either distribution is entirely zero
            if true_sum == 0 or pred_sum == 0:
                skipped_samples += 1
                continue

            # Normalize each sample independently
            true_vec = true_vec / true_sum
            pred_vec = pred_vec / pred_sum

            js_distance = jensenshannon(true_vec, pred_vec, base=2)
            js_divergence = js_distance ** 2

            sample_jsd.append(js_divergence)

        n_samples = len(sample_jsd)

        if n_samples == 0:
            print(f"Skipping {method}, Noise {noise_level}, Matrix {matrix}: no valid samples.")
            continue

        mean_jsd = np.mean(sample_jsd)
        std_jsd = np.std(sample_jsd)
        min_jsd = np.min(sample_jsd)
        max_jsd = np.max(sample_jsd)

        print(
            f"{method}, Noise {noise_level}, Matrix {matrix} — "
            f"N = {n_samples}, Skipped = {skipped_samples}, "
            f"Mean JSD = {mean_jsd:.6f}, SD = {std_jsd:.6f}, "
            f"Min = {min_jsd:.6f}, Max = {max_jsd:.6f}"
        )

        print(f"First 5 sample JSDs: {sample_jsd[:5]}")

        results.append([
            matrix,
            method,
            noise_level,
            mean_jsd,
            std_jsd,
            min_jsd,
            max_jsd,
            n_samples,
            skipped_samples
        ])

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# === SAVE SUMMARY TABLE ===
if results:
    results_df = pd.DataFrame(
        results,
        columns=[
            "Matrix",
            "Method",
            "NoiseLevel",
            "Mean_JS_divergence",
            "Std_JS_divergence",
            "Min_JS_divergence",
            "Max_JS_divergence",
            "N_samples",
            "Skipped_samples"
        ]
    )

    results_csv = os.path.join(results_dir, "Random_500_JSD_Summary.csv")

    # Overwrite instead of append, to avoid mixing old and new results
    results_df.to_csv(results_csv, index=False)

    print(f"Saved fresh JSD summary table: {results_csv}")

else:
    print("No results to save.")

In [14]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# === FONT SETTINGS ===
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'Results-JSD-PredictedVSActual/Random_500_JSD_Summary.csv'
df = pd.read_csv(file_path)

# === CREATE METHOD × NOISE MATRIX ===
pivot_df = df.pivot_table(
    index='Method',
    columns='NoiseLevel',
    values='Mean_JS_divergence',
    aggfunc='mean'
).sort_index(axis=1)

# === ORDER METHODS ===
method_order = [
    "BayesPrism",
    "MuSiC",
    "nuSVR",
    "CIBERSORTx",
    "NNLS",
    "QP",
    "ReDeconv"
]

pivot_df = pivot_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 4), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    pivot_df,
    cmap='viridis_r',             
    annot=True,
    fmt=".2f",
    annot_kws={"size": 10, "weight": "normal"},  # consistent with MAE heatmap
    cbar_kws={'shrink': 0.9},
    linewidths=0,
    linecolor='white'
)

# === AXES STYLING ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Noise Level', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# === COLORBAR STYLING ===
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Mean Jensen–Shannon divergence", fontsize=13, rotation=270, labelpad=20)
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === TICK STYLING ===
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# === SAVE ===
fig.savefig("Mean_JSD_Heatmap_Best-Matrix_Noise_500.svg")
plt.close(fig)

In [15]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# === FONT SETTINGS ===
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'Results-JSD-PredictedVSActual/Random_500_JSD_Summary.csv'
df = pd.read_csv(file_path)

# === CREATE METHOD × NOISE MATRIX ===
pivot_df = df.pivot_table(
    index='Method',
    columns='NoiseLevel',
    values='Mean_JS_divergence',
    aggfunc='mean'
).sort_index(axis=1)

# === ORDER METHODS ===
method_order = [
    "BayesPrism",
    "MuSiC",
    "nuSVR",
    "CIBERSORTx",
    "NNLS",
    "QP",
    "ReDeconv"
]

pivot_df = pivot_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 4), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    pivot_df,
    cmap='viridis_r',             
#    annot=True,
#    fmt=".2f",
#    annot_kws={"size": 10, "weight": "normal"},  # consistent with MAE heatmap
    cbar_kws={'shrink': 0.9},
    linewidths=0,
    linecolor='white'
)

# === AXES STYLING ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Noise Level', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# === COLORBAR STYLING ===
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Mean Jensen–Shannon divergence", fontsize=13, rotation=270, labelpad=20)
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === TICK STYLING ===
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# === SAVE ===
fig.savefig("Mean_JSD_Heatmap_Best-Matrix_Clean_Noise_500.svg")
plt.close(fig)